# Word Embeddings (Word2Vec) & Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from scipy.spatial.distance import cosine
from sklearn.manifold import TSNE
import os

df = pd.read_csv('../data/dataset_enriched.csv')
texts = df['cleaned_text'].dropna().astype(str).tolist()[:10000]
sentences = [t.split() for t in texts]

## 1. Word2Vec Training & File Check

In [ ]:
if os.path.exists('../data/word2vec.model'):
    print('Loading existing model...')
    model = Word2Vec.load('../data/word2vec.model')
else:
    print('Training Word2Vec...')
    model = Word2Vec(sentences, vector_size=100, window=5, min_count=5)
    model.save('../data/word2vec.model')

## 2. Similarity and Distances

In [ ]:
word = 'assurance'
if word in model.wv.key_to_index:
    print(f"Top 5 most similar words to '{word}':")
    for w, sim in model.wv.most_similar(word, topn=5):
        print(f" - {w}: {sim:.4f}")
        
    word2 = 'prix'
    if word2 in model.wv.key_to_index:
        dist = cosine(model.wv[word], model.wv[word2])
        print(f"\nCosine distance between '{word}' and '{word2}': {dist:.4f}")

## 3. Visualization (t-SNE)

In [ ]:
words = list(model.wv.key_to_index.keys())[:200]
vectors = np.array([model.wv[w] for w in words])

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(vectors)

plt.figure(figsize=(14, 10))
plt.scatter(tsne_results[:, 0], tsne_results[:, 1], alpha=0.6, color='b')
for i, word in enumerate(words):
    plt.annotate(word, xy=(tsne_results[i, 0], tsne_results[i, 1]), ha='right', va='bottom', fontsize=9)
plt.title('Word2Vec Visualisation with t-SNE')
plt.show()